### 20 Newsgroup 토픽 모델링

**20개 중 8개의 주제 데이터 로드 및 Count기반 피처 벡터화. LDA는 Count기반 Vectorizer만 적용합니다**

In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 모토사이클, 야구, 그래픽스, 윈도우즈, 중동, 기독교, 전자공학, 의학 등 8개 주제를 추출.
cats = ['rec.motorcycles', 'rec.sport.baseball', 'comp.graphics', 'comp.windows.x',
        'talk.politics.mideast', 'soc.religion.christian', 'sci.electronics', 'sci.med'  ]

# 위에서 cats 변수로 기재된 category만 추출. featch_20newsgroups( )의 categories에 cats 입력
news_df = fetch_20newsgroups(subset='all',remove=('headers', 'footers', 'quotes'),
                            categories=cats, random_state=0)

#LDA 는 Count기반의 Vectorizer만 적용합니다.
count_vect = CountVectorizer(max_df=0.95, max_features=1000, min_df=2, stop_words='english', ngram_range=(1,2))
feat_vect = count_vect.fit_transform(news_df.data)
print('CountVectorizer Shape:', feat_vect.shape)

CountVectorizer Shape: (7862, 1000)


**LDA 객체 생성 후 Count 피처 벡터화 객체로 LDA수행**

In [2]:
lda = LatentDirichletAllocation(n_components=8, random_state=0)
lda.fit(feat_vect)

LatentDirichletAllocation(n_components=8, random_state=0)

**각 토픽 모델링 주제별 단어들의 연관도 확인**  
lda객체의 components_ 속성은 주제별로 개별 단어들의 연관도 정규화 숫자가 들어있음

shape는 주제 개수 X 피처 단어 개수  

components_ 에 들어 있는 숫자값은 각 주제별로 단어가 나타난 횟수를 정규화 하여 나타냄.   

숫자가 클 수록 토픽에서 단어가 차지하는 비중이 높음  

In [3]:
print(lda.components_.shape)
lda.components_

(8, 1000)


array([[4.76246946e+01, 1.30590464e+02, 1.92572328e+01, ...,
        3.39537619e+01, 7.42649755e+01, 7.17392109e+01],
       [1.25919475e-01, 1.94924852e+00, 1.25043526e-01, ...,
        1.06955618e+02, 1.25080664e-01, 9.79579308e+01],
       [3.49904576e+02, 1.25142960e-01, 1.52107210e+02, ...,
        1.25118746e-01, 3.58796514e+01, 1.25025882e-01],
       ...,
       [2.40981635e+01, 2.04404394e+01, 8.55554560e+00, ...,
        2.35736693e+01, 6.47339848e+00, 1.55639443e+01],
       [1.25082857e-01, 1.27714385e-01, 1.25009975e-01, ...,
        1.25052378e+02, 1.25124695e-01, 4.03879645e+01],
       [1.25121653e-01, 1.77786583e+00, 1.25095618e-01, ...,
        1.86886165e+02, 1.25059007e-01, 1.25135497e-01]])

**각 토픽별 중심 단어 확인**

In [5]:
def display_topic_words(model, feature_names, no_top_words):
    for topic_index, topic in enumerate(model.components_):
        print('\nTopic #',topic_index)

        # components_ array에서 가장 값이 큰 순으로 정렬했을 때, 그 값의 array index를 반환.
        topic_word_indexes = topic.argsort()[::-1]
        top_indexes=topic_word_indexes[:no_top_words]

        # top_indexes대상인 index별로 feature_names에 해당하는 word feature 추출 후 join으로 concat
        feature_concat = ' '.join([str(feature_names[i]) for i in top_indexes])
        #feature_concat = ' + '.join([str(feature_names[i])+'*'+str(round(topic[i],1)) for i in top_indexes])
        print(feature_concat)

# CountVectorizer객체내의 전체 word들의 명칭을 get_features_names( )를 통해 추출
feature_names = count_vect.get_feature_names_out()

# Topic별 가장 연관도가 높은 word를 15개만 추출
display_topic_words(lda, feature_names, 15)

# 모토사이클, 야구, 그래픽스, 윈도우즈, 중동, 기독교, 전자공학, 의학 등 8개 주제를 추출.


Topic # 0
year 10 game medical health years 12 team 1993 20 disease cancer games patients good

Topic # 1
know don just said like people didn time did say think going went got right

Topic # 2
image file jpeg program color gif output images format files entry bit 00 use 03

Topic # 3
armenian armenians turkish people turkey government armenia 000 genocide turks muslim greek russian war university

Topic # 4
israel jews dos jewish israeli dos dos arab state arabs water palestinian people ed peace palestinians

Topic # 5
edu graphics com available window server ftp software mail use data information version windows sun

Topic # 6
god people jesus think church believe does christ say christian don know christians just bible

Topic # 7
like just use don good ve time make think problem used way does bike need


**개별 문서별 토픽 분포 확인**

lda객체의 transform()을 수행하면 개별 문서별 토픽 분포를 반환함.

In [6]:
doc_topics = lda.transform(feat_vect)
print(doc_topics.shape)
print(doc_topics[:3])

(7862, 8)
[[0.01389697 0.90263052 0.01389145 0.01390963 0.01389211 0.01390752
  0.01393413 0.01393766]
 [0.49706085 0.34366077 0.00212184 0.00212174 0.00212061 0.0021225
  0.00212225 0.14866944]
 [0.00544277 0.00544155 0.15160613 0.0054401  0.00544011 0.00544649
  0.0054399  0.81574294]]


**개별 문서별 토픽 분포도를 출력**

20newsgroup으로 만들어진 문서명을 출력.

fetch_20newsgroups()으로 만들어진 데이터의 filename속성은 모든 문서의 문서명을 가지고 있음.

filename속성은 절대 디렉토리를 가지는 문서명을 가지고 있으므로 '\\'로 분할하여 맨 마지막 두번째 부터 파일명으로 가져옴

In [7]:
def get_filename_list(newsdata):
    filename_list=[]

    for file in newsdata.filenames:
            #print(file)
            filename_temp = file.split('\\')[-2:]
            filename = '.'.join(filename_temp)
            filename_list.append(filename)

    return filename_list

filename_list = get_filename_list(news_df)
print("filename 개수:",len(filename_list), "filename list 10개만:",filename_list[:10])

filename 개수: 7862 filename list 10개만: ['/root/scikit_learn_data/20news_home/20news-bydate-train/soc.religion.christian/20630', '/root/scikit_learn_data/20news_home/20news-bydate-test/sci.med/59422', '/root/scikit_learn_data/20news_home/20news-bydate-test/comp.graphics/38765', '/root/scikit_learn_data/20news_home/20news-bydate-test/comp.graphics/38810', '/root/scikit_learn_data/20news_home/20news-bydate-test/sci.med/59449', '/root/scikit_learn_data/20news_home/20news-bydate-train/comp.graphics/38461', '/root/scikit_learn_data/20news_home/20news-bydate-train/comp.windows.x/66959', '/root/scikit_learn_data/20news_home/20news-bydate-train/rec.motorcycles/104487', '/root/scikit_learn_data/20news_home/20news-bydate-train/sci.electronics/53875', '/root/scikit_learn_data/20news_home/20news-bydate-train/sci.electronics/53617']


**DataFrame으로 생성하여 문서별 토픽 분포도 확인**

In [9]:
import pandas as pd

topic_names = ['Topic #'+ str(i) for i in range(0, 8)]
doc_topic_df = pd.DataFrame(data=doc_topics, columns=topic_names, index=filename_list)
doc_topic_df.head(3)

,Topic #0,Topic #1,Topic #2,Topic #3,Topic #4,Topic #5,Topic #6,Topic #7
/root/scikit_learn_data/20news_home/20news-bydate-train/soc.religion.christian/20630,0.013897,0.902631,0.013891,0.013910,0.013892,0.013908,0.013934,0.013938
/root/scikit_learn_data/20news_home/20news-bydate-test/sci.med/59422,0.497061,0.343661,0.002122,0.002122,0.002121,0.002123,0.002122,0.148669
/root/scikit_learn_data/20news_home/20news-bydate-test/comp.graphics/38765,0.005443,0.005442,0.151606,0.005440,0.005440,0.005446,0.005440,0.815743
